# Day 30：端到端 RAG 问答系统（核心日 · v0.3.0-day30）

> **🎙️ 讲师旁白**：欢迎来到星辰数科《智枢企业级智能体平台》第三阶段第 30 天！今天我们将把文档切分、向量入库、Top-K 检索、防幻觉 Prompt 构造、引用溯源（`Citation`）以及低相似度安全拒答（`我不知道/知识库中未找到相关信息`）完整串联，打造可落地的企业级 RAG 问答引擎。

## 1. 环境初始化与导入 `project_iterations/day_30` 模块
定位仓库根目录并将 `project_iterations/day_30` 加入 `sys.path`，导入 `Document`、`RAGPipeline` 与核心流水线函数。

In [1]:
import sys
from pathlib import Path

repo_root = Path.cwd()
while not (repo_root / "project_iterations").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
day30_dir = repo_root / "project_iterations" / "day_30"
if str(day30_dir) not in sys.path:
    sys.path.insert(0, str(day30_dir))

from zhishu_agent import Document, RAGPipeline

print("已加载模块路径:", day30_dir)

已加载模块路径: /home/antigravity/ab6539763/-AI-Agent/project_iterations/day_30


## 2. 文档切分入库（`ingest_documents`）与带引用问答（`answer_with_citations`）
初始化 `RAGPipeline`，写入企业 HR、财务与 IT 运维文档，测试知识库命中时的引用溯源输出。

In [2]:
pipeline = RAGPipeline(min_score_threshold=0.15, default_k=2)
docs = [
    Document(
        id="hr-001",
        page_content="星辰数科年假制度：入职满1年员工每年享有5天带薪年假，满10年享有10天带薪年假。",
        metadata={"source": "《星辰数科员工手册》", "department": "hr"},
    ),
    Document(
        id="fin-001",
        page_content="差旅报销规范：一线城市酒店住宿报销上限为每晚600元，需提供电子发票。",
        metadata={"source": "《财务差旅报销制度》", "department": "finance"},
    ),
    Document(
        id="it-001",
        page_content="Redis 生产运维规范：禁止执行 KEYS * 命令以免阻塞主线程，请统一使用 SCAN 遍历。",
        metadata={"source": "《SRE 运维手册》", "department": "it"},
    ),
]

chunk_ids = pipeline.ingest_documents(docs, chunk_size=120, chunk_overlap=20)
print("入库切片 IDs:", chunk_ids)

res = pipeline.answer_with_citations("员工带薪年假有几天？")
print(f"grounded={res.grounded} | max_score={res.max_score:.4f}")
print(res.answer)
for c in res.citations:
    print(f"  [{c.index}] {c.source} ({c.doc_id}, score={c.score:.4f})")

入库切片 IDs: ['hr-001', 'fin-001', 'it-001']
grounded=True | max_score=0.4425
根据企业知识库检索结果，为您整理如下：
- 星辰数科年假制度：入职满1年员工每年享有5天带薪年假，满10年享有10天带薪年假。 [1]
- Redis 生产运维规范：禁止执行 KEYS * 命令以免阻塞主线程，请统一使用 SCAN 遍历。 [2]
  [1] 《星辰数科员工手册》 (hr-001, score=0.4425)
  [2] 《SRE 运维手册》 (it-001, score=0.0882)


## 3. 低相似度安全拒答兜底与构造 Prompt 检查
当提问超出企业知识库范围（`max_score < min_score_threshold`）时，系统应返回 `grounded=False` 与标准拒答提示。

In [3]:
out_of_domain = pipeline.answer_with_citations("火星基地的班车几点发车？")
print(f"grounded={out_of_domain.grounded} | max_score={out_of_domain.max_score:.4f}")
print("citations:", out_of_domain.citations)
print("answer:", out_of_domain.answer)

grounded=False | max_score=0.1164
citations: []
answer: 我不知道/知识库中未找到相关信息。当前最高相似度为 0.1164，低于阈值 0.1500，已触发安全拒答。
